In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load and inspect the tab-separated data.
train_df = pd.read_csv(train_file_path, sep="\t", header=None, names=["label", "message"])
test_df = pd.read_csv(test_file_path, sep="\t", header=None, names=["label", "message"])

# Convert labels to 0 for ham and 1 for spam.
train_labels = (train_df["label"] == "spam").astype(int)
test_labels = (test_df["label"] == "spam").astype(int)


In [ ]:
# Train a word and character n-gram classifier. Character features help with
# misspellings and unusual spam formatting; the word model captures phrases.
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

features = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)),
    ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),
                             sublinear_tf=True, min_df=2, max_features=150000)),
])
model = Pipeline([
    ("features", features),
    ("classifier", LogisticRegression(C=8, max_iter=1000, class_weight="balanced")),
])
model.fit(train_df["message"], train_labels)


In [ ]:
# The second value is the model's spam probability.
def predict_message(pred_text):
  spam_probability = float(model.predict_proba([pred_text])[0][1])
  label = "spam" if spam_probability >= 0.5 else "ham"
  return [spam_probability, label]

pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
